# Spatially Masked Regression on AJILE12: surrogates, lagged model and full masking

Ostadsharif Memar & Dehghani (2026), arXiv:2606.11415, on DANDI:000055 version 0.220127.0436
(the only published version; see `reproduce_memar_2026.ipynb` for the header, the version choice
and the full data check). Claims tested here: surrogate-ieeg, surrogate-order-ieeg,
lagged-ieeg-no-gain and full-mask-above-zero-ieeg (which uses the mask sweep's m = 1.0 values; the
m = 1.0 run is repeated here with this notebook's generator). Settings and deviations as in
`reproduce_memar_2026.ipynb` (events from `ReachEvents`); the last section shows how much they
change the score on one subject. The surrogate and full-mask claims are run under both readings of where the preprocessing is
applied, (a) before segmentation and (b) per epoch; the end-matched phase surrogate and the
lagged model under reading (a) only, for time.

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"          # one linear-algebra thread per process; the fits run in parallel
import time, zlib, json, resource
import numpy as np
import pandas as pd
from scipy import stats
import smr

T0 = time.time()
PAPER = "memar-2026"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

DECIM, MAX_EPOCH = 10, 200
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 13)]
if os.environ.get("SMR_TEST"):                 # development only: two subjects
    SUBJECTS = ["sub-07", "sub-04"]
IDX = [int(s[-2:]) - 1 for s in SUBJECTS]
print("dataset", smr.DANDISET, smr.VERSION)

## Data check

The trials of every subject, preprocessed as in `reproduce_memar_2026.ipynb` under both readings (its data check covers the metadata, duplicates and the clock).

In [ ]:
t = time.time()
DATA = {}
for s in SUBJECTS:
    S = smr.load_subject(s)
    DATA[s] = smr.drop_uncoordinated(S)
    x = DATA[s]["a"]
    print(s, "sessions", len(S["sessions"]), "trials", x.shape[0], "electrodes", x.shape[1], "samples", x.shape[2],
          "dropped without coordinates", DATA[s]["n_dropped_el"], f"range {x.min():.2f} to {x.max():.2f}, SD {x.std():.3f}",
          f"reading b range {DATA[s]['b'].min():.2f} to {DATA[s]['b'].max():.2f} ({time.time() - t:.0f} s)", flush=True)
print("trials in all", sum(S["a"].shape[0] for S in DATA.values()))
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

In [ ]:
def run_all(m, rng, reading="a", workers=3, **kw):
    # one SMR condition on every subject, in parallel; each subject's generator is seeded from rng
    seeds = rng.integers(2**63, size=len(SUBJECTS))
    jobs = [(smr.run_smr, (DATA[s][reading], smr.neighbours(DATA[s]["xyz"]), m, np.random.default_rng(k)),
             dict(decim=DECIM, max_epoch=MAX_EPOCH, **kw)) for s, k in zip(SUBJECTS, seeds)]
    return dict(zip(SUBJECTS, smr.parallel(jobs, workers)))

def subj(res):
    return np.array([res[s]["dc"].mean() for s in SUBJECTS])

def msd(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"

## Instantaneous model at m = 1.0

The baseline for the surrogate and lagged comparisons, and the m = 1.0 values for
full-mask-above-zero-ieeg: the common pipeline at m = 1.0 under both preprocessing readings, with
the generators of lagged-ieeg-no-gain (the claim's for reading a, its second for reading b; the
lagged runs below continue from them).

In [ ]:
t = time.time()
RNG_LAG = {"a": np.random.default_rng(seed("lagged-ieeg-no-gain")),
           "b": np.random.default_rng(seed("lagged-ieeg-no-gain") + 1)}
INST, inst = {}, {}
for prep in ("a", "b"):
    INST[prep] = run_all(1.0, RNG_LAG[prep], prep)
    inst[prep] = subj(INST[prep])
    print(f"reading {prep}, instantaneous m = 1.0: {msd(inst[prep])} ({time.time() - t:.0f} s)", flush=True)
    print(dict(zip(SUBJECTS, inst[prep].round(3))))

## surrogate-ieeg

> On iEEG surrogates, intra-subject DistCorr falls from 0.553 to 0.181 (phase shuffle), 0.232
> (IAAFT) and 0.201 (block shuffle).

Surrogates of every preprocessed trial and channel, then the common pipeline at m = 1.0 (the
claim's generator for preprocessing reading a, its second for reading b, each seeding one
generator per subject for the surrogate draws and the fit). Both readings of the phase surrogate:
(a) as the code, the whole 2 s trial; (b) end-matched (run with preprocessing reading a only, for time), each series cut to the 1.8 s segment whose
end values and end slopes match best (`smr.end_matched`), then shuffled. Phase shuffling one
series at a time already breaks the timing between channels, so each series takes its own segment
and the trials keep one length.

In [ ]:
TAB_SUR = {"phase": (0.181, 0.029), "phase_end": (0.181, 0.029), "iaaft": (0.232, 0.060), "block": (0.201, 0.168)}
KINDS = ("phase", "phase_end", "iaaft", "block")
t = time.time()
SUR = {}
for prep, add in (("a", 0), ("b", 1)):
    rng_sur = np.random.default_rng(seed("surrogate-ieeg") + add)
    SUR[prep] = {}
    for kind in (KINDS if prep == "a" else ("phase", "iaaft", "block")):
        seeds = rng_sur.integers(2**63, size=len(SUBJECTS))
        jobs = [(smr.run_surrogate, (DATA[s][prep], kind, smr.neighbours(DATA[s]["xyz"]), 1.0, np.random.default_rng(k)),
                 dict(decim=DECIM, max_epoch=MAX_EPOCH)) for s, k in zip(SUBJECTS, seeds)]
        res = dict(zip(SUBJECTS, smr.parallel(jobs, 3)))
        v = SUR[prep][kind] = subj(res)
        exp, tol = TAB_SUR[kind]
        print(f"reading {prep}, {kind}: {msd(v)}; within {exp} ± {tol}: {abs(v.mean() - exp) <= tol}; below the original "
              f"({inst[prep].mean():.3f}): {v.mean() < inst[prep].mean()}; subjects below their original: "
              f"{int((v < inst[prep]).sum())} of {len(SUBJECTS)}; median epoch of the best validation loss "
              f"{int(np.median(np.concatenate([res[s]['best_ep'] for s in SUBJECTS])))} ({time.time() - t:.0f} s)", flush=True)
for prep in ("a", "b"):
    print(f"--- preprocessing reading {prep}")
    print(pd.DataFrame({"subject": SUBJECTS, "original": inst[prep], **SUR[prep]}).round(3).to_string(index=False))
    for phase in [k for k in ("phase", "phase_end") if k in SUR[prep]]:
        ks = (phase, "iaaft", "block")
        ok = all(abs(SUR[prep][k].mean() - TAB_SUR[k][0]) <= TAB_SUR[k][1] and SUR[prep][k].mean() < inst[prep].mean() for k in ks)
        print(f"   with {phase}: all three within tolerance and below the original: {ok}")

**Result: reproduced.** Under preprocessing reading (a) the averages are 0.202 (phase), 0.196 (end-matched phase), 0.184 (IAAFT) and 0.230 (block); under reading (b) 0.207, 0.192 and 0.232. Each is within its tolerance and below the original (0.602 and 0.571), in every subject.

## surrogate-order-ieeg

> In iEEG, phase shuffling degrades reconstruction most on average, and IAAFT surrogates retain
> somewhat more reconstructability.

In [ ]:
for prep in ("a", "b"):
    for phase in [k for k in ("phase", "phase_end") if k in SUR[prep]]:
        avgs = {k: round(float(SUR[prep][k].mean()), 3) for k in (phase, "iaaft", "block")}
        v = SUR[prep]
        print(f"preprocessing {prep}, phase surrogate {phase}: averages {avgs}; phase lowest of the three: "
              f"{min(avgs, key=avgs.get) == phase}; IAAFT above phase (reading a): {avgs['iaaft'] > avgs[phase]}; "
              f"phase < block < IAAFT (reading b): {avgs[phase] < avgs['block'] < avgs['iaaft']}; subjects with phase lowest: "
              f"{int(sum(v[phase][i] < min(v['iaaft'][i], v['block'][i]) for i in range(len(SUBJECTS))))} of {len(SUBJECTS)}")

**Result: not reproduced.** IAAFT, not phase shuffling, gives the lowest average under every reading run: 0.184 against 0.202 (phase) and 0.196 (end-matched phase) with preprocessing (a), 0.192 against 0.207 with (b). Phase is lowest in 0 of 12 subjects in each, and the order of Table II (reading b of the claim) does not hold. End matching lowers the phase average by 0.006, less than its gap to IAAFT.

## full-mask-above-zero-ieeg

> With the whole local neighborhood masked, iEEG reconstruction remains above zero, so the target
> signal is not determined by its immediate neighborhood alone.

The criterion takes the m = 1.0 values of the mask sweep; they are recomputed here (the run above)
to compare them with the phase-shuffled surrogate in one notebook. `masking_coverage.ipynb` prints
the sweep's own m = 1.0 values.

In [ ]:
for prep in ("a", "b"):
    print(f"preprocessing {prep}: (a) m = 1.0 average {inst[prep].mean():.3f} > 0: {inst[prep].mean() > 0}; smallest electrode value "
          f"{min(INST[prep][s]['dc'].min() for s in SUBJECTS):.3f}")
    for phase in [k for k in ("phase", "phase_end") if k in SUR[prep]]:
        print(f"   (b) against the {phase} surrogate average {SUR[prep][phase].mean():.3f}: above {inst[prep].mean() > SUR[prep][phase].mean()}; "
              f"subjects above their own surrogate value: {int((inst[prep] > SUR[prep][phase]).sum())} of {len(SUBJECTS)}")

**Result: reproduced (on the criterion's assumed reading b).** With the whole neighbourhood masked the average is 0.602 (a) and 0.571 (b), above zero (reading a of the criterion) and above the phase-shuffled average, 0.202 or 0.196 end-matched (a) and 0.207 (b), in 12 of 12 subjects.

## lagged-ieeg-no-gain

> Adding lagged predictors (0, 20, 30, 50 and 60 ms) barely changes iEEG reconstruction (0.554 ±
> 0.067 against 0.553 ± 0.068).

Preprocessing reading (a) only; the lagged model under reading (b) was not run, for time.

In [ ]:
t = time.time()
print("lag offsets in samples:", smr.lag_samples())
LAG, lag = {}, {}
for prep in ("a",):                 # reading b not run, for time
    LAG[prep] = run_all(1.0, RNG_LAG[prep], prep, lagged=True)
    lag[prep] = subj(LAG[prep])
    d = lag[prep] - inst[prep]
    print(f"--- preprocessing reading {prep}: lagged {msd(lag[prep])} against instantaneous {msd(inst[prep])} ({time.time() - t:.0f} s)")
    print(f"lagged average within 0.554 ± 0.067: {abs(lag[prep].mean() - 0.554) <= 0.067}; "
          f"difference {lag[prep].mean() - inst[prep].mean():+.4f}, {100 * (lag[prep].mean() - inst[prep].mean()) / inst[prep].mean():+.1f}% of the "
          f"instantaneous average; under 10%: {abs(lag[prep].mean() - inst[prep].mean()) < 0.1 * inst[prep].mean()}")
    print("per subject lagged minus instantaneous:", dict(zip(SUBJECTS, d.round(4))))
    print("median epoch of the best validation loss, lagged:", [int(np.median(LAG[prep][s]["best_ep"])) for s in SUBJECTS])
    if len(SUBJECTS) > 5:
        w = stats.wilcoxon(lag[prep], inst[prep])
        print(f"context only: Wilcoxon signed-rank over subjects, statistic {w.statistic}, p = {w.pvalue:.4f}; "
              f"subjects with a gain {int((d > 0).sum())} of {len(SUBJECTS)}")

**Result: reproduced.** The lagged average is 0.604 ± 0.059 against 0.602 ± 0.057 for the instantaneous model (preprocessing reading a), within 0.554 ± 0.067 and a difference of +0.3% of the instantaneous average; 7 of 12 subjects gain, Wilcoxon p = 0.79. Reading (b) was not run, for time.

## How much the speed deviations change the score

On sub-07, preprocessing reading (a), the m = 1.0 model is trained closer to the paper's code
(every sample, early stopping with a patience of 200 epochs, up to 10000 epochs) in two parts, for
time: (1) every 10th sample with early stopping up to 3000 epochs, (2) every sample for 100
epochs, against this attempt's setting (every 10th sample, 200 epochs). The lagged model, the
Local model and a phase surrogate fit are checked with early stopping up to 1000 epochs. Every run
uses the fourth generator of intra-ieeg-mean (its seed plus 3).

In [ ]:
S = DATA["sub-07"]
nbr = smr.neighbours(S["xyz"])
g = seed("intra-ieeg-mean") + 3
t = time.time()
runs = [("instantaneous", "this attempt: every 10th sample, 200 epochs", smr.run_smr, (S["a"], nbr, 1.0), dict(decim=10, max_epoch=200)),
        ("instantaneous", "every 10th sample, up to 3000 epochs, patience 200", smr.run_smr, (S["a"], nbr, 1.0), dict(decim=10, max_epoch=3000)),
        ("instantaneous", "every sample, 100 epochs", smr.run_smr, (S["a"], nbr, 1.0), dict(decim=1, max_epoch=100))]
for model, kw in (("lagged", dict(lagged=True)), ("local", dict(mode="local"))):
    m = 0.0 if model == "local" else 1.0
    runs += [(model, "this attempt: every 10th sample, 200 epochs", smr.run_smr, (S["a"], nbr, m), dict(decim=10, max_epoch=200, **kw)),
             (model, "every 10th sample, up to 1000 epochs, patience 200", smr.run_smr, (S["a"], nbr, m), dict(decim=10, max_epoch=1000, **kw))]
runs += [("phase surrogate", "this attempt: every 10th sample, 200 epochs", smr.run_surrogate, (S["a"], "phase", nbr, 1.0), dict(decim=10, max_epoch=200)),
         ("phase surrogate", "every 10th sample, up to 1000 epochs, patience 200", smr.run_surrogate, (S["a"], "phase", nbr, 1.0), dict(decim=10, max_epoch=1000))]
jobs = [(fn, args + (np.random.default_rng(g),), kw) for _, _, fn, args, kw in runs]
res = smr.parallel(jobs, 4)
rows = [dict(model=mo, setting=lab, mean_distcorr=round(r["dc"].mean(), 4), epochs_run=r["n_ep"],
             median_best_epoch=int(np.median(r["best_ep"]))) for (mo, lab, *_), r in zip(runs, res)]
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
for mo, grp in chk.groupby("model", sort=False):
    print(f"{mo}: largest difference from this attempt's setting {(grp.mean_distcorr - grp.mean_distcorr.iloc[0]).abs().max():.4f}")
print(f"({time.time() - t:.0f} s)")

## Summary

| Claim | Outcome |
|---|---|
| surrogate-ieeg | reproduced (0.202, 0.184, 0.230 under preprocessing a; 0.207, 0.192, 0.232 under b) |
| surrogate-order-ieeg | not reproduced (IAAFT lowest under every reading run, not phase) |
| full-mask-above-zero-ieeg | reproduced, on the criterion's assumed reading (0.602 against 0.202) |
| lagged-ieeg-no-gain | reproduced (0.604 against 0.602, preprocessing a) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method. The speed deviations change the sub-07
score by at most 0.0002 for every model checked. Not run, for time: the end-matched phase
surrogate under preprocessing reading (b) and the lagged model under reading (b). The other
differences and the dataset findings are listed in the summary of `reproduce_memar_2026.ipynb`.


In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")